# BEV Lab 2 — BEVFusion: Camera + LiDAR Fusion

In Lab 1 we ran **LSS** — a camera-only model that lifts 2D image features into a 3D frustum and splats them onto a BEV grid.

In this lab we add **LiDAR**. BEVFusion (Liu et al., 2022) builds a BEV representation from each modality independently, then fuses them by concatenation:

```
Camera images  →  LSS-style lift+splat  →  Camera BEV  ─┐
                                                         ├─ concat → conv head → 3D boxes / BEV seg
LiDAR points   →  VoxelNet / PointPillars →  LiDAR BEV  ─┘
```

We'll load three pretrained checkpoints on the same nuScenes keyframe:
1. **Camera-only** — what does the camera see alone?
2. **LiDAR-only** — what does the LiDAR see alone?
3. **Fused (camera + LiDAR)** — what does combining both give you?

> Reference: *BEVFusion: Multi-Task Multi-Sensor Fusion with Unified Bird's-Eye View Representation* — Liu et al., ICRA 2023.  
> Code: [mit-han-lab/bevfusion](https://github.com/mit-han-lab/bevfusion)

## Part 1 — Setup

In [ ]:
# Downloads: BEVFusion repo + pretrained weights + nuScenes mini
!pip install -q nuscenes-devkit pyquaternion
!git clone -q https://github.com/mit-han-lab/bevfusion.git

# 3 checkpoints: camera-only, lidar-only, and fused (each ~150-300 MB)
!mkdir -p bevfusion/pretrained
!wget -qO bevfusion/pretrained/camera-only-seg.pth  'https://www.dropbox.com/scl/fi/cwpcu80n0shmwraegi6z4/camera-only-seg.pth?rlkey=l60kdaz19fq3gwocsjk09e60z'
!wget -qO bevfusion/pretrained/lidar-only-seg.pth   'https://www.dropbox.com/scl/fi/mi3w6uxvytdre9i42r9k7/lidar-only-seg.pth?rlkey=rve7hx80u3en1gfoi7tjucl72'
!wget -qO bevfusion/pretrained/bevfusion-seg.pth    'https://www.dropbox.com/scl/fi/8lgd1hkod2a15mwry0fvd/bevfusion-seg.pth?rlkey=2tmgw7mcrlwy9qoqeui63tay9'

# nuScenes mini (same as Lab 1 — skip if already downloaded)
![ -d nuscenes-mini/v1.0-mini ] || (mkdir -p nuscenes-mini && wget -q https://www.nuscenes.org/data/v1.0-mini.tgz -O- | tar -xz -C nuscenes-mini)
![ -d nuscenes-mini/maps/expansion ] || (wget -qO /tmp/mapexp.zip https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/nuScenes-map-expansion-v1.3.zip && unzip -qo /tmp/mapexp.zip -d nuscenes-mini/maps)

In [ ]:
# Install BEVFusion's dependencies (pinned versions from their repo)
!cd bevfusion && pip install -q -e . 2>&1 | tail -3
!pip install -q mmcv-full==1.4.0 mmdet==2.20.0 torchpack 2>&1 | tail -3

In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, 'bevfusion')
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from nuscenes.nuscenes import NuScenes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nusc   = NuScenes('v1.0-mini', dataroot='nuscenes-mini', verbose=False)
print(f'device: {device}')
print(f'nuScenes mini: {len(nusc.sample)} samples across {len(nusc.scene)} scenes')

## Part 2 — Load one sample: 6 cameras + LiDAR

Unlike Lab 1 (cameras only), BEVFusion also needs the **LiDAR point cloud**. A nuScenes keyframe has:
- 6 surround camera images (1600×900)
- 1 LiDAR sweep from LIDAR_TOP (~35k points, x/y/z/intensity/ring)

In [ ]:
from nuscenes.utils.data_classes import LidarPointCloud
from pyquaternion import Quaternion
import os

# Pick a sample
sample = nusc.sample[10]

# Load the 6 camera images
cam_names = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']
cam_images = {}
for cam in cam_names:
    sd = nusc.get('sample_data', sample['data'][cam])
    cam_images[cam] = Image.open(os.path.join(nusc.dataroot, sd['filename']))

# Load the LiDAR point cloud and transform to ego frame
lidar_sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
pc = LidarPointCloud.from_file(os.path.join(nusc.dataroot, lidar_sd['filename']))

calib = nusc.get('calibrated_sensor', lidar_sd['calibrated_sensor_token'])
pc.rotate(Quaternion(calib['rotation']).rotation_matrix)
pc.translate(np.array(calib['translation']))

lidar_points = pc.points.T  # (N, 5) — x, y, z, intensity, ring

print(f'Cameras: {len(cam_images)} images, each {cam_images["CAM_FRONT"].size}')
print(f'LiDAR:   {lidar_points.shape[0]:,} points')
print(f'  x range: {lidar_points[:,0].min():.1f} .. {lidar_points[:,0].max():.1f} m')
print(f'  z range: {lidar_points[:,2].min():.1f} .. {lidar_points[:,2].max():.1f} m')

In [ ]:
# Show what we have: 6 cameras + LiDAR BEV (top-down scatter of points)
fig, axes = plt.subplots(3, 3, figsize=(14, 10))

# Top row + middle row: 6 cameras
for i, cam in enumerate(cam_names):
    ax = axes[i // 3, i % 3]
    ax.imshow(cam_images[cam]); ax.axis('off')
    ax.set_title(cam, fontsize=9)

# Bottom-center: LiDAR bird's eye view
for ax in axes[2]: ax.axis('off')
ax_lid = axes[2, 1]
ax_lid.scatter(lidar_points[:, 1], lidar_points[:, 0],
               c=lidar_points[:, 2], s=0.3, cmap='turbo', vmin=-2, vmax=3)
ax_lid.set_xlim(50, -50); ax_lid.set_ylim(-50, 50)
ax_lid.set_aspect('equal')
ax_lid.set_title(f'LiDAR top-down ({lidar_points.shape[0]:,} points, colored by height)', fontsize=9)
ax_lid.scatter([0], [0], c='red', marker='^', s=100, zorder=5)

plt.tight_layout(); plt.show()

## Part 3 — Run BEVFusion inference

We'll run three variants on the exact same keyframe:
1. **Camera-only** — only the 6 images, no LiDAR
2. **LiDAR-only** — only the point cloud, no cameras
3. **Fused** — both modalities combined

Each produces a BEV segmentation map (drivable area, lanes, vehicles, etc.).

*Coming next — loading the BEVFusion configs and running inference...*